# 01 · Exploratory data analysis
Run `python -m src.pipeline` first. This notebook reuses the tested pipeline modules, so every number here matches the report.
The same analysis is written to `docs/eda_report.md` on every pipeline run.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config
cfg = load_config()
feats = pd.read_csv(cfg.path('processed_dir') / 'customer_features_train.csv.gz', index_col=0)
meta = json.load(open(cfg.path('artifacts_dir') / 'meta.json'))
feats.shape, round(feats.churn.mean(), 3)

## 1. Distributions (RFM)
Expect heavy right skew in frequency and spend; this is why the neural nets and clustering use a signed log transform.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, c in zip(ax, ['recency_days', 'frequency', 'monetary_total']):
    feats[c].clip(upper=feats[c].quantile(.99)).hist(ax=a, bins=30); a.set_title(c)
plt.tight_layout()

## 2. What relates to churn?

In [ ]:
cols = meta['model_columns']
feats[cols].corrwith(feats.churn).sort_values(key=abs, ascending=False).head(10)

In [ ]:
feats.groupby(pd.cut(feats.recency_days, [0, 30, 60, 120, 240, 480, 2000]), observed=True).churn.mean().plot.bar(title='Churn rate by recency bucket')

## 3. Multicollinearity (VIF)
Exact-collinear groups (category shares sum to 1; country dummies sum to 1) and near-duplicates (`monetary_total` vs `clv_historical`) are pruned iteratively.

In [ ]:
from src.features.eda import vif_table
all_cols = meta['all_columns']
print('VIF before pruning (top 8):'); print(vif_table(feats[all_cols]).head(8).round(1))
print('\nVIF after pruning (top 5):'); print(vif_table(feats[cols]).head(5).round(1))

## 4. Hypotheses carried into modelling
Generated from the data in `docs/eda_report.md`; read them there so they always match the latest run.

In [ ]:
print(open(cfg.path('docs_dir') / 'eda_report.md').read().split('## Hypotheses')[-1])